In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
op_df = pd.read_csv("cleaned_data/operational_points.csv", sep=";")
station_to_station_df = pd.read_csv("cleaned_data/station_to_station.csv", sep=";")

In [3]:
op_df.info()
op_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              1337 non-null   int64  
 1   Geo_x           1320 non-null   float64
 2   Geo_y           1320 non-null   float64
 3   Code_TAF_TAP    1337 non-null   object 
 4   Classification  1337 non-null   object 
 5   Symbolic_name   1336 non-null   object 
 6   Name_FR_short   1337 non-null   object 
 7   Name_FR_full    1337 non-null   object 
 8   Name_NL_short   1337 non-null   object 
 9   Name_NL_full    1337 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 104.6+ KB


,ID,Geo_x,Geo_y,Code_TAF_TAP,Classification,Symbolic_name,Name_FR_short,Name_FR_full,Name_NL_short,Name_NL_full
0,443,50.923363,5.497283,BE00443,Station,FKGLF,Genk-Zd-Ford,Genk-Zuid-L.O.-Ford,Genk-Zd-Ford,Genk-Zuid-L.O.-Ford
1,275,50.400574,4.458827,BE00275,Service installation,GCRA,Charleroi-AT,Charleroi-A.T.,Charleroi-AT,Charleroi-A.T.
2,1748,51.306684,4.293916,BE01748,Connection,VOIL1,V.Oiltank1,Verb.Oiltanking 1,V.Oiltank1,Verb.Oiltanking 1
3,1749,51.308302,4.290241,BE01749,Connection,VOIL2,V.Oiltank2,Verb.Oiltanking 2,V.Oiltank2,Verb.Oiltanking 2
4,1751,51.335505,4.283473,BE01751,Connection,VIBR,V.IBR,Verb.IBR,V.IBR,Verb.IBR


In [ ]:
def parse_coordinates(coord_str):
	try:
		coords = json.loads(coord_str)
		if isinstance(coords, list) and all(isinstance(c, list) for c in coords):
			return [[float(x) for x in c] for c in coords]
	except Exception:
		print(f"Invalid coordinate format: {coord_str}")
	return np.nan

station_to_station_df["Coordinates"] = station_to_station_df["Coordinates"].apply(parse_coordinates)


In [5]:
station_to_station_df.info()
station_to_station_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1385 entries, 0 to 1384
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Departure_station_id  1385 non-null   int64  
 1   Arrival_station_id    1385 non-null   int64  
 2   Geo_x                 1385 non-null   float64
 3   Geo_y                 1385 non-null   float64
 4   Distance              1385 non-null   float64
 5   Coordinates           1385 non-null   object 
dtypes: float64(3), int64(2), object(1)
memory usage: 65.1+ KB


,Departure_station_id,Arrival_station_id,Geo_x,Geo_y,Distance,Coordinates
0,841,821,50.943277,4.221739,2.46,"[[50.932897, 4.216581], [50.933034, 4.216732],..."
1,246,958,50.520529,3.559228,4.96,"[[50.513613, 3.592063], [50.513585, 3.590017],..."
2,457,672,50.732192,4.505572,1.74,"[[50.73811, 4.497448], [50.737018, 4.498736], ..."
3,782,77,50.621327,3.788608,2.34,"[[50.614049, 3.800495], [50.61668, 3.796664], ..."
4,1102,592,50.526327,5.226778,1.18,"[[50.527376, 5.234037], [50.526833, 5.232868],..."


In [ ]:
# On récupère toutes les stations de départ et d'arrivée
stations = pd.concat([
	station_to_station_df[["Departure_station_id"]],
	station_to_station_df[["Arrival_station_id"]].rename(columns={"Arrival_station_id": "Departure_station_id"})
], ignore_index=True).drop_duplicates()

found_ids = stations["Departure_station_id"].isin(op_df["ID"])
found_stations = found_ids.sum()
nb_stations = len(stations)
missing_ids = stations.loc[~found_ids, "Departure_station_id"]

for station_id in missing_ids:
	print(f"Station ID {station_id} not found in operational points")

print(f"PTCAR_IDs found: {found_stations} / {nb_stations} ({found_stations / nb_stations:.2%}%)")


Station ID 864 not found in operational points
PTCAR_IDs found: 558 / 559 (99.82%%)


In [ ]:
station_to_station_df[
	(station_to_station_df["Departure_station_id"] == 864) |
	(station_to_station_df["Arrival_station_id"] == 864)
].head()

,Departure_station_id,Arrival_station_id,Geo_x,Geo_y,Distance,Coordinates
176,864,866,51.177225,4.451503,1.30,"[[51.171808, 4.455389], [51.172885, 4.454764],..."
672,866,864,51.177225,4.451503,1.30,"[[51.171808, 4.455389], [51.172885, 4.454764],..."
783,864,139,51.190584,4.439749,2.12,"[[51.182317, 4.447188], [51.18316, 4.446394], ..."
1240,139,864,51.190584,4.439749,2.12,"[[51.182317, 4.447188], [51.18316, 4.446394], ..."


In [8]:
op_df[op_df["Name_FR_full"] == "Mortsel"].head()

,ID,Geo_x,Geo_y,Code_TAF_TAP,Classification,Symbolic_name,Name_FR_short,Name_FR_full,Name_NL_short,Name_NL_full
150,863,51.182891,4.448812,BE00863,Stop in open track,GMO,Mortsel,Mortsel,Mortsel,Mortsel


In [ ]:
df = station_to_station_df.copy()

# 1) Extraire les points de début/fin depuis la polyline "Coordinates" (liste de [x, y])
df["start"] = df["Coordinates"].str[0]    # premier point
df["end"]   = df["Coordinates"].str[-1]   # dernier point

# Séparer x/y pour start et end (deux colonnes chacune)
df[["start_x", "start_y"]] = pd.DataFrame(df["start"].tolist(), index=df.index)
df[["end_x",   "end_y"  ]] = pd.DataFrame(df["end"].tolist(),   index=df.index)

# 2) Préparer la table des points opérationnels pour jointure
ops = op_df[["ID", "Geo_x", "Geo_y"]].copy()

# 3) Joindre les coordonnées de la station de départ (par ID)
df = df.merge(
	ops.rename(columns={"Geo_x": "dep_x", "Geo_y": "dep_y"}),
	left_on="Departure_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 4) Joindre les coordonnées de la station d'arrivée (par ID)
df = df.merge(
	ops.rename(columns={"Geo_x": "arr_x", "Geo_y": "arr_y"}),
	left_on="Arrival_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 5) Comparer vectoriellement :
# - Une station (dep ou arr) est valide si ses coords correspondent au start OU au end de la polyline
dep_matches_start = (df["dep_x"].eq(df["start_x"])) & (df["dep_y"].eq(df["start_y"]))
dep_matches_end   = (df["dep_x"].eq(df["end_x"]))   & (df["dep_y"].eq(df["end_y"]))
dep_valid = dep_matches_start | dep_matches_end

arr_matches_start = (df["arr_x"].eq(df["start_x"])) & (df["arr_y"].eq(df["start_y"]))
arr_matches_end   = (df["arr_x"].eq(df["end_x"]))   & (df["arr_y"].eq(df["end_y"]))
arr_valid = arr_matches_start | arr_matches_end

both_valid = dep_valid & arr_valid

# 6) Comptages
nb_total_stations = len(df)
valid_departure_coords = int(dep_valid.sum())
valid_arrival_coords   = int(arr_valid.sum())
valid_both_coords      = int(both_valid.sum())

print(f"Valid departure coordinates: {valid_departure_coords} / {nb_total_stations} ({valid_departure_coords / nb_total_stations:.2%}%)")
print(f"Valid arrival coordinates: {valid_arrival_coords} / {nb_total_stations} ({valid_arrival_coords / nb_total_stations:.2%}%)")
print(f"Valid both coordinates: {valid_both_coords} / {nb_total_stations} ({valid_both_coords / nb_total_stations:.2%}%)")

Valid departure coordinates: 1381 / 1385 (99.71%%)
Valid arrival coordinates: 1381 / 1385 (99.71%%)
Valid both coordinates: 1377 / 1385 (99.42%%)


In [ ]:
# 1️⃣ Obtenir toutes les stations (départ + arrivée)
stations_temp = pd.concat([
	station_to_station_df[["Departure_station_id"]],
	station_to_station_df[["Arrival_station_id"]].rename(columns={"Arrival_station_id": "Departure_station_id"})
], ignore_index=True).drop_duplicates()

# 2️⃣ Joindre avec les points opérationnels pour récupérer les infos
stations_df = stations_temp.merge(
	op_df[["ID", "Geo_x", "Geo_y", "Symbolic_name", "Name_FR_short", "Name_FR_full"]],
	left_on="Departure_station_id",
	right_on="ID",
	how="left"
).drop_duplicates()

# 3️⃣ Sélectionner les colonnes finales
stations_df = stations_df[["ID", "Geo_x", "Geo_y", "Symbolic_name", "Name_FR_short", "Name_FR_full"]].drop_duplicates()

# 4️⃣ Affichage
print(len(stations_df))
stations_df.info()
stations_df.head(10)

559
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 559 entries, 0 to 558
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ID             558 non-null    float64
 1   Geo_x          558 non-null    float64
 2   Geo_y          558 non-null    float64
 3   Symbolic_name  557 non-null    object 
 4   Name_FR_short  558 non-null    object 
 5   Name_FR_full   558 non-null    object 
dtypes: float64(3), object(3)
memory usage: 26.3+ KB


,ID,Geo_x,Geo_y,Symbolic_name,Name_FR_short,Name_FR_full
0,841.0,50.932897,4.216581,LHL,Mollem,Mollem
1,246.0,50.527312,3.526278,FCA,Callenelle,Callenelle
2,457.0,50.726468,4.513842,MGV,Genval,Genval
3,782.0,50.614049,3.800495,FFA,Maffle,Maffle
4,1102.0,50.528178,5.219617,LHY,Statte,Statte
5,157.0,50.868680,5.509372,FIE,Bilzen,Bilzen
6,1912.0,50.809668,4.398413,GAR,Arcades,Arcades
7,143.0,51.088643,5.234551,MBV,Beverlo,Beverlo
8,901.0,50.571986,5.741746,FNV,Nessonvaux,Nessonvaux
9,1761.0,50.819202,4.404357,FDEL,Delta,Delta


In [ ]:
error = 1e-3

# 1) Repart d'une copie de travail
df = station_to_station_df.copy()

# 2) Extraire start/end (les extrémités de la polyline "Coordinates")
df["start"] = df["Coordinates"].str[0]
df["end"]   = df["Coordinates"].str[-1]

# Décomposer en x/y
df[["start_x", "start_y"]] = pd.DataFrame(df["start"].tolist(), index=df.index)
df[["end_x",   "end_y"  ]] = pd.DataFrame(df["end"].tolist(),   index=df.index)

# 3) Préparer la table des stations (issue de stations_df)
stations_min = stations_df[["ID", "Geo_x", "Geo_y"]].drop_duplicates()

# 4) Joindre les coordonnées de la station de départ
df = df.merge(
	stations_min.rename(columns={"Geo_x": "dep_x", "Geo_y": "dep_y"}),
	left_on="Departure_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 5) Joindre les coordonnées de la station d’arrivée
df = df.merge(
	stations_min.rename(columns={"Geo_x": "arr_x", "Geo_y": "arr_y"}),
	left_on="Arrival_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 6) Comparaisons tolérantes (à +/- error)
dep_matches_start = np.isclose(df["dep_x"], df["start_x"], atol=error) & np.isclose(df["dep_y"], df["start_y"], atol=error)
dep_matches_end   = np.isclose(df["dep_x"], df["end_x"],   atol=error) & np.isclose(df["dep_y"], df["end_y"],   atol=error)
dep_valid = dep_matches_start | dep_matches_end

arr_matches_start = np.isclose(df["arr_x"], df["start_x"], atol=error) & np.isclose(df["arr_y"], df["start_y"], atol=error)
arr_matches_end   = np.isclose(df["arr_x"], df["end_x"],   atol=error) & np.isclose(df["arr_y"], df["end_y"],   atol=error)
arr_valid = arr_matches_start | arr_matches_end

both_valid = dep_valid & arr_valid

# 7) Comptages
nb_total_stations = len(df)
valid_departure_coords = int(dep_valid.sum())
valid_arrival_coords   = int(arr_valid.sum())
valid_both_coords      = int(both_valid.sum())


# 8) Logs identiques au code d’origine pour les paires non vérifiées
#    (affiche chaque paire avec True/False pour départ/arrivée)
not_ok = df.loc[~both_valid, ["Departure_station_id", "Arrival_station_id"]].copy()
not_ok["dep_ok"] = dep_valid[~both_valid]
not_ok["arr_ok"] = arr_valid[~both_valid]

for row in not_ok.itertuples(index=False):
	print(f"Could not verify coordinates for station pair "
		f"{row.Departure_station_id} ({row.dep_ok}) -> {row.Arrival_station_id} ({row.arr_ok})")
		
print(f"Valid departure coordinates: {valid_departure_coords} / {nb_total_stations} ({valid_departure_coords / nb_total_stations:.2%}%)")
print(f"Valid arrival coordinates: {valid_arrival_coords} / {nb_total_stations} ({valid_arrival_coords / nb_total_stations:.2%}%)")
print(f"Valid both coordinates: {valid_both_coords} / {nb_total_stations} ({valid_both_coords / nb_total_stations:.2%}%)")



Could not verify coordinates for station pair 864 (False) -> 866 (True)
Could not verify coordinates for station pair 866 (True) -> 864 (False)
Could not verify coordinates for station pair 864 (False) -> 139 (True)
Could not verify coordinates for station pair 837 (False) -> 1081 (True)
Could not verify coordinates for station pair 1081 (True) -> 837 (False)
Could not verify coordinates for station pair 837 (False) -> 128 (True)
Could not verify coordinates for station pair 128 (True) -> 837 (False)
Could not verify coordinates for station pair 139 (True) -> 864 (False)
Valid departure coordinates: 1381 / 1385 (99.71%%)
Valid arrival coordinates: 1381 / 1385 (99.71%%)
Valid both coordinates: 1377 / 1385 (99.42%%)


In [ ]:
station_to_station_df[
	(station_to_station_df["Departure_station_id"] == 837) |
	(station_to_station_df["Arrival_station_id"] == 837)
].head()

,Departure_station_id,Arrival_station_id,Geo_x,Geo_y,Distance,Coordinates
793,837,1081,50.786158,4.344922,2.81,"[[50.793586, 4.36043], [50.791152, 4.353316], ..."
794,1081,837,50.786158,4.344922,2.81,"[[50.793586, 4.36043], [50.791152, 4.353316], ..."
1223,837,128,50.772798,4.315710,2.36,"[[50.777891, 4.329808], [50.777405, 4.328658],..."
1224,128,837,50.772798,4.315710,2.36,"[[50.777891, 4.329808], [50.777405, 4.328658],..."


In [ ]:
station_to_station_df[
	(station_to_station_df["Departure_station_id"] == 864) |
	(station_to_station_df["Arrival_station_id"] == 864)
].head()

,Departure_station_id,Arrival_station_id,Geo_x,Geo_y,Distance,Coordinates
176,864,866,51.177225,4.451503,1.30,"[[51.171808, 4.455389], [51.172885, 4.454764],..."
672,866,864,51.177225,4.451503,1.30,"[[51.171808, 4.455389], [51.172885, 4.454764],..."
783,864,139,51.190584,4.439749,2.12,"[[51.182317, 4.447188], [51.18316, 4.446394], ..."
1240,139,864,51.190584,4.439749,2.12,"[[51.182317, 4.447188], [51.18316, 4.446394], ..."


In [14]:
stations_df[(stations_df["Name_FR_full"] == "Saint-Job") | (stations_df["Name_FR_full"] == "Moensberg")].head()

,ID,Geo_x,Geo_y,Symbolic_name,Name_FR_short,Name_FR_full
325,1081.0,50.793586,4.360430,GSO,Saint-Job,Saint-Job
486,837.0,50.779471,4.333527,MOE,Moensberg,Moensberg


In [ ]:
new_station = pd.DataFrame([{"ID": 864,
	"Geo_x" : 51.182317,
	"Geo_y" : 4.447188,
	"Symbolic_name": "GMOD",
	"Name_FR_short": "Deurnesteenweg",
	"Name_FR_full": "Mortsel-Deurnesteenweg"}])
stations_df = pd.concat([stations_df, new_station], ignore_index=True)

In [16]:
stations_df[(stations_df["Name_FR_full"] == "Mortsel-Deurnesteenweg")].head()

,ID,Geo_x,Geo_y,Symbolic_name,Name_FR_short,Name_FR_full
559,864.0,51.182317,4.447188,GMOD,Deurnesteenweg,Mortsel-Deurnesteenweg


In [ ]:
# On récupère toutes les stations de départ et d'arrivée
stations = pd.concat([
	station_to_station_df[["Departure_station_id"]],
	station_to_station_df[["Arrival_station_id"]].rename(columns={"Arrival_station_id": "Departure_station_id"})
], ignore_index=True).drop_duplicates()

found_ids = stations["Departure_station_id"].isin(stations_df["ID"])
found_stations = found_ids.sum()
nb_stations = len(stations)
missing_ids = stations.loc[~found_ids, "Departure_station_id"]

for station_id in missing_ids:
	print(f"Station ID {station_id} not found in operational points")

print(f"PTCAR_IDs found: {found_stations} / {nb_stations} ({found_stations / nb_stations:.2%}%)")

PTCAR_IDs found: 559 / 559 (100.00%%)


In [ ]:
mask = (
	((station_to_station_df["Departure_station_id"] == 837) & (station_to_station_df["Arrival_station_id"] == 128)) |
	((station_to_station_df["Departure_station_id"] == 128) & (station_to_station_df["Arrival_station_id"] == 837))
)

def replace_first(coords):
	return [[50.779471, 4.333527]] + coords[1:]

station_to_station_df.loc[mask, "Coordinates"] = (
	station_to_station_df.loc[mask, "Coordinates"].apply(replace_first)
)

In [ ]:
mask = (
	((station_to_station_df["Departure_station_id"] == 837) & (station_to_station_df["Arrival_station_id"] == 1081)) |
	((station_to_station_df["Departure_station_id"] == 1081) & (station_to_station_df["Arrival_station_id"] == 837))
)

def replace_first(coords):
	return coords[:-1] + [[50.779471, 4.333527]] 

station_to_station_df.loc[mask, "Coordinates"] = (
	station_to_station_df.loc[mask, "Coordinates"].apply(replace_first)
)

In [20]:
station_to_station_df[mask].head()

,Departure_station_id,Arrival_station_id,Geo_x,Geo_y,Distance,Coordinates
793,837,1081,50.786158,4.344922,2.81,"[[50.793586, 4.36043], [50.791152, 4.353316], ..."
794,1081,837,50.786158,4.344922,2.81,"[[50.793586, 4.36043], [50.791152, 4.353316], ..."


In [21]:
station_to_station_df.loc[mask, "Coordinates"].apply(lambda ll: ll[0]).head()

793    [50.793586, 4.36043]
794    [50.793586, 4.36043]
Name: Coordinates, dtype: object

In [ ]:
error = 1e-6

# 1) Repart d'une copie de travail
df = station_to_station_df.copy()

# 2) Extraire start/end (les extrémités de la polyline "Coordinates")
df["start"] = df["Coordinates"].str[0]
df["end"]   = df["Coordinates"].str[-1]

# Décomposer en x/y
df[["start_x", "start_y"]] = pd.DataFrame(df["start"].tolist(), index=df.index)
df[["end_x",   "end_y"  ]] = pd.DataFrame(df["end"].tolist(),   index=df.index)

# 3) Préparer la table des stations (issue de stations_df)
stations_min = stations_df[["ID", "Geo_x", "Geo_y"]].drop_duplicates()

# 4) Joindre les coordonnées de la station de départ
df = df.merge(
	stations_min.rename(columns={"Geo_x": "dep_x", "Geo_y": "dep_y"}),
	left_on="Departure_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 5) Joindre les coordonnées de la station d’arrivée
df = df.merge(
	stations_min.rename(columns={"Geo_x": "arr_x", "Geo_y": "arr_y"}),
	left_on="Arrival_station_id", right_on="ID", how="left"
).drop(columns=["ID"])

# 6) Comparaisons tolérantes (à +/- error)
dep_matches_start = np.isclose(df["dep_x"], df["start_x"], atol=error) & np.isclose(df["dep_y"], df["start_y"], atol=error)
dep_matches_end   = np.isclose(df["dep_x"], df["end_x"],   atol=error) & np.isclose(df["dep_y"], df["end_y"],   atol=error)
dep_valid = dep_matches_start | dep_matches_end

arr_matches_start = np.isclose(df["arr_x"], df["start_x"], atol=error) & np.isclose(df["arr_y"], df["start_y"], atol=error)
arr_matches_end   = np.isclose(df["arr_x"], df["end_x"],   atol=error) & np.isclose(df["arr_y"], df["end_y"],   atol=error)
arr_valid = arr_matches_start | arr_matches_end

both_valid = dep_valid & arr_valid

# 7) Comptages
nb_total_stations = len(df)
valid_departure_coords = int(dep_valid.sum())
valid_arrival_coords   = int(arr_valid.sum())
valid_both_coords      = int(both_valid.sum())


# 8) Logs identiques au code d’origine pour les paires non vérifiées
#    (affiche chaque paire avec True/False pour départ/arrivée)
not_ok = df.loc[~both_valid, ["Departure_station_id", "Arrival_station_id", 
	"start_x", "dep_x", "start_y", "dep_y", "arr_x", "arr_y", "end_x", "end_y"]].copy()
not_ok["dep_ok"] = dep_valid[~both_valid]
not_ok["arr_ok"] = arr_valid[~both_valid]

for row in not_ok.itertuples(index=False):
	print(f"Could not verify coordinates for station pair "
		f"{row.Departure_station_id} ({row.dep_ok}) -> {row.Arrival_station_id} ({row.arr_ok})")
	print(f"  start: ({row.start_x}, {row.start_y}) vs dep: ({row.dep_x}, {row.dep_y})")
	print(f"  start: ({row.start_x}, {row.start_y}) vs arr: ({row.arr_x}, {row.arr_y})")
	print(f"  end: ({row.end_x}, {row.end_y}) vs dep: ({row.dep_x}, {row.dep_y})")
	print(f"  end: ({row.end_x}, {row.end_y}) vs arr: ({row.arr_x}, {row.arr_y})")

print(f"Valid departure coordinates: {valid_departure_coords} / {nb_total_stations} ({valid_departure_coords / nb_total_stations:.2%}%)")
print(f"Valid arrival coordinates: {valid_arrival_coords} / {nb_total_stations} ({valid_arrival_coords / nb_total_stations:.2%}%)")
print(f"Valid both coordinates: {valid_both_coords} / {nb_total_stations} ({valid_both_coords / nb_total_stations:.2%}%)")

Valid departure coordinates: 1385 / 1385 (100.00%%)
Valid arrival coordinates: 1385 / 1385 (100.00%%)
Valid both coordinates: 1385 / 1385 (100.00%%)


In [23]:
stations_df[stations_df["Symbolic_name"].isna()].head()

,ID,Geo_x,Geo_y,Symbolic_name,Name_FR_short,Name_FR_full
159,NaN,NaN,NaN,NaN,NaN,NaN
399,24.0,50.539174,5.288862,NaN,Ampsin,Ampsin


In [24]:
stations_df.loc[stations_df["Name_FR_short"] == "Ampsin", "Symbolic_name"] = "AMPS"
stations_df = stations_df.dropna()

In [25]:
stations_df = stations_df.dropna()
stations_df["ID"] = stations_df["ID"].astype(int) 

In [27]:
stations_df.to_csv("sumo_data/stations.csv", sep=";", index=False)
station_to_station_df.to_csv("sumo_data/station_to_station.csv", sep=";", index=False)